# Additional Metrics Analysis

This notebook compares additional calibration metrics (Brier Score, Categorical NLL, AURC) across different datasets and loss functions for the journal revision.

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Configuration

Define all experiments to compare across datasets.

In [22]:
# Define all experiments
experiments = {
    'ACDC': [
        'bundles/acdc17_baseline_dice_ce_2/seed_12345/inference_results_additional',
        'bundles/acdc17_hardl1ace_dice_ce_2/seed_12345/inference_results_additional',
        'bundles/acdc17_softl1ace_dice_ce_2/seed_12345/inference_results_additional',
        'bundles/acdc17_baseline_ce_2/seed_12345/inference_results_additional',
        'bundles/acdc17_baseline_dice_ce_2/seed_12345_temp_scaled/inference_results_additional',
        'bundles/acdc17_hardl1ace_dice_ce_2/seed_12345_temp_scaled/inference_results_additional',
        'bundles/acdc17_softl1ace_dice_ce_2/seed_12345_temp_scaled/inference_results_additional',
    ],
    'AMOS': [
        'bundles/amos22_baseline_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/amos22_hardl1ace_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/amos22_softl1ace_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/amos22_baseline_ce_nl/seed_12345/inference_results_additional',
        'bundles/amos22_baseline_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
        'bundles/amos22_hardl1ace_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
        'bundles/amos22_softl1ace_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
    ],
    'BraTS': [
        'bundles/brats21_baseline_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/brats21_hardl1ace_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/brats21_softl1ace_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/brats21_baseline_ce_nl/seed_12345/inference_results_additional',
        'bundles/brats21_baseline_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
        'bundles/brats21_hardl1ace_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
        'bundles/brats21_softl1ace_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
    ],
    'KiTS': [
        'bundles/kits23_baseline_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/kits23_hardl1ace_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/kits23_softl1ace_dice_ce_nl/seed_12345/inference_results_additional',
        'bundles/kits23_baseline_ce_nl/seed_12345/inference_results_additional',
        'bundles/kits23_baseline_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
        'bundles/kits23_hardl1ace_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
        'bundles/kits23_softl1ace_dice_ce_nl/seed_12345_temp_scaled/inference_results_additional',
    ]
}

# Method names for display (matching paper format)
method_names = [
    'Dice + CE (Baseline)',
    'Dice + CE + hL1-ACE',
    'Dice + CE + sL1-ACE',
    'CE Only',
    'Dice + CE + Ts',
    'Dice + CE + hL1-ACE + Ts',
    'Dice + CE + sL1-ACE + Ts'
]

# Metrics to analyze
metrics = ['brier_score', 'categorical_nll', 'aurc']

# Base path
base_path = Path('/workspaces/Average-Calibration-Losses')

## Data Loading Functions

In [23]:
from scipy import stats

def load_metric_raw(csv_path):
    """
    Load raw metric CSV and return per-sample values.
    
    Args:
        csv_path: Path to the raw CSV file
        
    Returns:
        Dictionary with 'mean', 'std', and 'values' keys
    """
    try:
        df = pd.read_csv(csv_path)
        
        # The 'mean' column contains the per-sample mean across classes
        if 'mean' in df.columns:
            values = df['mean'].values
            return {
                'mean': np.nanmean(values),
                'std': np.nanstd(values, ddof=1),
                'values': values  # Keep raw values for statistical testing
            }
        else:
            print(f"Warning: No 'mean' column found in {csv_path}")
            return {'mean': np.nan, 'std': np.nan, 'values': np.array([])}
    except FileNotFoundError:
        print(f"Warning: File not found: {csv_path}")
        return {'mean': np.nan, 'std': np.nan, 'values': np.array([])}
    except Exception as e:
        print(f"Error loading {csv_path}: {e}")
        return {'mean': np.nan, 'std': np.nan, 'values': np.array([])}


def load_all_metrics(experiment_path, metrics):
    """
    Load all metrics for a single experiment.
    
    Args:
        experiment_path: Path to the inference_results_additional directory
        metrics: List of metric names
        
    Returns:
        Dictionary mapping metric names to {'mean': float, 'std': float, 'values': array}
    """
    results = {}
    exp_path = base_path / experiment_path
    
    for metric in metrics:
        csv_file = exp_path / f"{metric}_raw.csv"
        results[metric] = load_metric_raw(csv_file)
    
    return results


def paired_t_test(values1, values2, alpha=0.05):
    """
    Perform paired t-test and return whether difference is significant.
    
    Args:
        values1: Array of values for method 1
        values2: Array of values for method 2
        alpha: Significance level (default 0.05)
        
    Returns:
        tuple: (is_significant, p_value, mean_diff)
    """
    if len(values1) == 0 or len(values2) == 0:
        return False, np.nan, np.nan
    
    # Remove NaN pairs
    mask = ~(np.isnan(values1) | np.isnan(values2))
    v1 = values1[mask]
    v2 = values2[mask]
    
    if len(v1) < 2:
        return False, np.nan, np.nan
    
    t_stat, p_value = stats.ttest_rel(v1, v2)
    mean_diff = np.mean(v1 - v2)
    
    return p_value < alpha, p_value, mean_diff

## Load All Results

In [24]:
# Load all results into a nested dictionary
all_results = {}

for dataset, paths in experiments.items():
    all_results[dataset] = {}
    for method_name, path in zip(method_names, paths):
        all_results[dataset][method_name] = load_all_metrics(path, metrics)

print("Data loading complete!")

Data loading complete!


## Create Comparison Tables with Statistical Significance

Tables show results with statistical significance testing (paired t-test, α=0.05). Bold values indicate statistically significantly better than baseline.

In [25]:
def format_scaled_value(value, scale_factor, n_decimals=2):
    """Format a value scaled by a factor (e.g., ×10^-3) with fixed decimal places."""
    if np.isnan(value):
        return 'N/A'
    scaled = value * scale_factor
    return f"{scaled:.{n_decimals}f}"


def create_metric_table_with_significance(metric_name, all_results, datasets, methods, baseline_method=None):
    """
    Create a formatted table for a specific metric with statistical significance testing.
    """
    if baseline_method is None:
        baseline_method = methods[0]
    
    # Determine scaling factor based on metric
    if metric_name == 'aurc':
        scale_factor = 100  # ×10^-2
    else:
        scale_factor = 1000  # ×10^-3
    
    data = []
    raw_values = {}
    
    for dataset in datasets:
        row = {'Dataset': dataset}
        raw_values[dataset] = {}
        baseline_values = all_results[dataset][baseline_method][metric_name]['values']
        
        for method in methods:
            result = all_results[dataset][method][metric_name]
            mean_val = result['mean']
            std_val = result['std']
            values = result['values']
            
            raw_values[dataset][method] = mean_val
            
            if np.isnan(mean_val):
                row[method] = 'N/A'
            else:
                mean_scaled = format_scaled_value(mean_val, scale_factor)
                std_scaled = format_scaled_value(std_val, scale_factor)
                formatted = f"{mean_scaled} ± {std_scaled}"
                
                if method != baseline_method:
                    is_sig, p_val, mean_diff = paired_t_test(baseline_values, values)
                    if is_sig and mean_diff > 0:
                        if p_val < 0.01:
                            formatted += " **"
                        else:
                            formatted += " *"
                
                row[method] = formatted
        
        data.append(row)
    
    df = pd.DataFrame(data)
    
    def highlight_best(row):
        styles = [''] * len(row)
        dataset_name = row['Dataset']
        numeric_values = {}
        for method in methods:
            val = raw_values.get(dataset_name, {}).get(method, np.nan)
            if not np.isnan(val):
                numeric_values[method] = val
        
        if numeric_values:
            best_method = min(numeric_values, key=numeric_values.get)
            best_idx = list(row.index).index(best_method)
            styles[best_idx] = 'font-weight: bold'
        
        return styles
    
    styled_df = df.style.apply(highlight_best, axis=1).set_properties(**{'text-align': 'left'}).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left')]},
        {'selector': 'td', 'props': [('text-align', 'left'), ('white-space', 'nowrap')]}
    ])
    
    return styled_df, df, raw_values


def create_latex_table_brier(all_results, datasets, methods, baseline_method=None):
    """Create LaTeX table for Brier Score with exact caption format."""
    if baseline_method is None:
        baseline_method = methods[0]
    
    scale_factor = 1000  # ×10^-3
    
    # Find best (minimum) value for each dataset
    best_methods = {}
    for dataset in datasets:
        min_val = float('inf')
        best_method = None
        for method in methods:
            mean_val = all_results[dataset][method]['brier_score']['mean']
            if not np.isnan(mean_val) and mean_val < min_val:
                min_val = mean_val
                best_method = method
        best_methods[dataset] = best_method
    
    latex = """\\begin{table}[H]
\\centering
\\caption{\\textbf{Comparison of Brier Score metric ($\\times 10^{-3}$, lower is better) between baseline, cross-entropy only and our auxiliary average calibration error (hL1-ACE and sL1-ACE) losses with and without temperature scaling (Ts). Statistical signifance of difference to baseline $^*$p $<$ 0.05, $^{**}$p $<$ 0.01. Best value per dataset is shown in bold.}}
\\begin{tabular}{lcccc}
\\hline
Method & ACDC & AMOS & BraTS & KiTS \\\\
\\hline
"""
    
    for method in methods:
        row_data = [method]
        for dataset in datasets:
            baseline_values = all_results[dataset][baseline_method]['brier_score']['values']
            result = all_results[dataset][method]['brier_score']
            mean_val = result['mean']
            std_val = result['std']
            values = result['values']
            
            if np.isnan(mean_val):
                formatted = 'N/A'
            else:
                mean_scaled = format_scaled_value(mean_val, scale_factor)
                std_scaled = format_scaled_value(std_val, scale_factor)
                formatted = f"{mean_scaled} ± {std_scaled}"
                
                if method != baseline_method:
                    is_sig, p_val, mean_diff = paired_t_test(baseline_values, values)
                    if is_sig and mean_diff > 0:
                        if p_val < 0.01:
                            formatted += "$^{**}$"
                        else:
                            formatted += "$^{*}$"
                
                if method == best_methods[dataset]:
                    formatted = f"\\textbf{{{formatted}}}"
            
            row_data.append(formatted)
        
        latex += " & ".join(row_data) + " \\\\\n"
    
    latex += """\\hline
\\end{tabular}
\\label{tab:brier_score}
\\end{table}
"""
    return latex


def create_latex_table_nll(all_results, datasets, methods, baseline_method=None):
    """Create LaTeX table for Categorical NLL with exact caption format."""
    if baseline_method is None:
        baseline_method = methods[0]
    
    scale_factor = 1000  # ×10^-3
    
    best_methods = {}
    for dataset in datasets:
        min_val = float('inf')
        best_method = None
        for method in methods:
            mean_val = all_results[dataset][method]['categorical_nll']['mean']
            if not np.isnan(mean_val) and mean_val < min_val:
                min_val = mean_val
                best_method = method
        best_methods[dataset] = best_method
    
    latex = """\\begin{table}[H]
\\centering
\\caption{\\textbf{Comparison of categorical negative log-likelihood (NLL) ($\\times 10^{-3}$, lower is better) across all datasets for baseline, CE-only training, and our auxiliary calibration losses (hL1-ACE and sL1-ACE), with and without post-hoc temperature scaling (Ts). Statistical significance is shown relative to the baseline: $^*$p $<$ 0.05, $^{**}$p $<$ 0.01. Best value per dataset is highlighted in bold.}}
\\begin{tabular}{lcccc}
\\hline
Method & ACDC $\\downarrow$ & AMOS $\\downarrow$ & BraTS $\\downarrow$ & KiTS $\\downarrow$ \\\\
\\hline
"""
    
    for method in methods:
        row_data = [method]
        for dataset in datasets:
            baseline_values = all_results[dataset][baseline_method]['categorical_nll']['values']
            result = all_results[dataset][method]['categorical_nll']
            mean_val = result['mean']
            std_val = result['std']
            values = result['values']
            
            if np.isnan(mean_val):
                formatted = 'N/A'
            else:
                mean_scaled = format_scaled_value(mean_val, scale_factor)
                std_scaled = format_scaled_value(std_val, scale_factor)
                formatted = f"{mean_scaled} ± {std_scaled}"
                
                if method != baseline_method:
                    is_sig, p_val, mean_diff = paired_t_test(baseline_values, values)
                    if is_sig and mean_diff > 0:
                        if p_val < 0.01:
                            formatted += "$^{**}$"
                        else:
                            formatted += "$^{*}$"
                
                if method == best_methods[dataset]:
                    formatted = f"\\textbf{{{formatted}}}"
            
            row_data.append(formatted)
        
        latex += " & ".join(row_data) + " \\\\\n"
    
    latex += """\\hline
\\end{tabular}
\\label{tab:categorical_nll}
\\end{table}
"""
    return latex


def create_latex_table_aurc(all_results, datasets, methods, baseline_method=None):
    """Create LaTeX table for AURC with exact caption format."""
    if baseline_method is None:
        baseline_method = methods[0]
    
    scale_factor = 100  # ×10^-2
    
    best_methods = {}
    for dataset in datasets:
        min_val = float('inf')
        best_method = None
        for method in methods:
            mean_val = all_results[dataset][method]['aurc']['mean']
            if not np.isnan(mean_val) and mean_val < min_val:
                min_val = mean_val
                best_method = method
        best_methods[dataset] = best_method
    
    latex = """\\begin{table}[H]
\\centering
\\caption{\\textbf{Comparison of area under the risk-coverage curve (AURC) ($\\times 10^{-2}$, lower is better) for all datasets using baseline, CE-only models, and our auxiliary calibration loss variants (hL1-ACE, sL1-ACE), with and without temperature scaling (Ts). Statistical significance relative to baseline is indicated: $^*$p $<$ 0.05, $^{**}$p $<$ 0.01. Best result per dataset is in bold.}}
\\begin{tabular}{lcccc}
\\hline
Method & ACDC $\\downarrow$ & AMOS $\\downarrow$ & BraTS $\\downarrow$ & KiTS $\\downarrow$ \\\\
\\hline
"""
    
    for method in methods:
        row_data = [method]
        for dataset in datasets:
            baseline_values = all_results[dataset][baseline_method]['aurc']['values']
            result = all_results[dataset][method]['aurc']
            mean_val = result['mean']
            std_val = result['std']
            values = result['values']
            
            if np.isnan(mean_val):
                formatted = 'N/A'
            else:
                mean_scaled = format_scaled_value(mean_val, scale_factor)
                std_scaled = format_scaled_value(std_val, scale_factor)
                formatted = f"{mean_scaled} ± {std_scaled}"
                
                if method != baseline_method:
                    is_sig, p_val, mean_diff = paired_t_test(baseline_values, values)
                    if is_sig and mean_diff > 0:
                        if p_val < 0.01:
                            formatted += "$^{**}$"
                        else:
                            formatted += "$^{*}$"
                
                if method == best_methods[dataset]:
                    formatted = f"\\textbf{{{formatted}}}"
            
            row_data.append(formatted)
        
        latex += " & ".join(row_data) + " \\\\\n"
    
    latex += """\\hline
\\end{tabular}
\\label{tab:aurc}
\\end{table}
"""
    return latex


# Create tables for each metric
datasets = list(experiments.keys())

print("="*100)
print("BRIER SCORE (×10^-3, Lower is Better)")
print("* indicates p < 0.05, ** indicates p < 0.01")
print("Best value per dataset is shown in bold")
print("="*100)
brier_table, brier_df, brier_raw = create_metric_table_with_significance('brier_score', all_results, datasets, method_names)
display(brier_table)
print("\nLaTeX Table:")
brier_latex = create_latex_table_brier(all_results, datasets, method_names)
print(brier_latex)
print()

print("="*100)
print("CATEGORICAL NLL (×10^-3, Lower is Better)")
print("* indicates p < 0.05, ** indicates p < 0.01")
print("Best value per dataset is shown in bold")
print("="*100)
nll_table, nll_df, nll_raw = create_metric_table_with_significance('categorical_nll', all_results, datasets, method_names)
display(nll_table)
print("\nLaTeX Table:")
nll_latex = create_latex_table_nll(all_results, datasets, method_names)
print(nll_latex)
print()

print("="*100)
print("AURC (×10^-2, Lower is Better)")
print("* indicates p < 0.05, ** indicates p < 0.01")
print("Best value per dataset is shown in bold")
print("="*100)
aurc_table, aurc_df, aurc_raw = create_metric_table_with_significance('aurc', all_results, datasets, method_names)
display(aurc_table)
print("\nLaTeX Table:")
aurc_latex = create_latex_table_aurc(all_results, datasets, method_names)
print(aurc_latex)

BRIER SCORE (×10^-3, Lower is Better)
* indicates p < 0.05, ** indicates p < 0.01
Best value per dataset is shown in bold


,Dataset,Dice + CE (Baseline),Dice + CE + hL1-ACE,Dice + CE + sL1-ACE,CE Only,Dice + CE + Ts,Dice + CE + hL1-ACE + Ts,Dice + CE + sL1-ACE + Ts
0,ACDC,2.79 ± 1.07,2.60 ± 0.92 **,2.67 ± 0.98 *,2.79 ± 1.05,2.62 ± 1.00 **,2.49 ± 0.86 **,2.64 ± 0.92 **
1,AMOS,0.48 ± 0.36,0.50 ± 0.42,0.51 ± 0.40,0.51 ± 0.45,0.46 ± 0.34 **,0.49 ± 0.40,0.51 ± 0.39
2,BraTS,0.65 ± 0.75,0.65 ± 0.87,0.77 ± 0.85,0.71 ± 0.94,0.63 ± 0.71 **,0.64 ± 0.83,0.77 ± 0.82
3,KiTS,0.55 ± 1.52,0.46 ± 0.70,0.68 ± 1.59,0.58 ± 1.48,0.62 ± 1.66,0.57 ± 0.64,0.67 ± 1.50



LaTeX Table:
\begin{table}[H]
\centering
\caption{\textbf{Comparison of Brier Score metric ($\times 10^{-3}$, lower is better) between baseline, cross-entropy only and our auxiliary average calibration error (hL1-ACE and sL1-ACE) losses with and without temperature scaling (Ts). Statistical signifance of difference to baseline $^*$p $<$ 0.05, $^{**}$p $<$ 0.01. Best value per dataset is shown in bold.}}
\begin{tabular}{lcccc}
\hline
Method & ACDC & AMOS & BraTS & KiTS \\
\hline
Dice + CE (Baseline) & 2.79 ± 1.07 & 0.48 ± 0.36 & 0.65 ± 0.75 & 0.55 ± 1.52 \\
Dice + CE + hL1-ACE & 2.60 ± 0.92$^{**}$ & 0.50 ± 0.42 & 0.65 ± 0.87 & \textbf{0.46 ± 0.70} \\
Dice + CE + sL1-ACE & 2.67 ± 0.98$^{*}$ & 0.51 ± 0.40 & 0.77 ± 0.85 & 0.68 ± 1.59 \\
CE Only & 2.79 ± 1.05 & 0.51 ± 0.45 & 0.71 ± 0.94 & 0.58 ± 1.48 \\
Dice + CE + Ts & 2.62 ± 1.00$^{**}$ & \textbf{0.46 ± 0.34$^{**}$} & \textbf{0.63 ± 0.71$^{**}$} & 0.62 ± 1.66 \\
Dice + CE + hL1-ACE + Ts & \textbf{2.49 ± 0.86$^{**}$} & 0.49 ± 0.40 & 0.64 

,Dataset,Dice + CE (Baseline),Dice + CE + hL1-ACE,Dice + CE + sL1-ACE,CE Only,Dice + CE + Ts,Dice + CE + hL1-ACE + Ts,Dice + CE + sL1-ACE + Ts
0,ACDC,6.68 ± 3.42,5.31 ± 2.50 **,5.26 ± 2.48 **,6.79 ± 3.75,5.10 ± 2.37 **,4.59 ± 1.81 **,4.93 ± 1.85 **
1,AMOS,1.13 ± 1.34,1.15 ± 1.55,1.14 ± 1.59,1.25 ± 1.91,0.97 ± 0.94 **,1.07 ± 1.09,1.08 ± 1.12
2,BraTS,3.04 ± 4.92,3.30 ± 6.66,3.44 ± 6.65,3.00 ± 5.68,2.57 ± 3.40 **,2.86 ± 4.65,3.58 ± 4.87
3,KiTS,2.56 ± 7.96,1.82 ± 2.83,2.99 ± 8.38,2.78 ± 8.73,2.38 ± 6.61,2.83 ± 2.21,3.15 ± 5.93



LaTeX Table:
\begin{table}[H]
\centering
\caption{\textbf{Comparison of categorical negative log-likelihood (NLL) ($\times 10^{-3}$, lower is better) across all datasets for baseline, CE-only training, and our auxiliary calibration losses (hL1-ACE and sL1-ACE), with and without post-hoc temperature scaling (Ts). Statistical significance is shown relative to the baseline: $^*$p $<$ 0.05, $^{**}$p $<$ 0.01. Best value per dataset is highlighted in bold.}}
\begin{tabular}{lcccc}
\hline
Method & ACDC $\downarrow$ & AMOS $\downarrow$ & BraTS $\downarrow$ & KiTS $\downarrow$ \\
\hline
Dice + CE (Baseline) & 6.68 ± 3.42 & 1.13 ± 1.34 & 3.04 ± 4.92 & 2.56 ± 7.96 \\
Dice + CE + hL1-ACE & 5.31 ± 2.50$^{**}$ & 1.15 ± 1.55 & 3.30 ± 6.66 & \textbf{1.82 ± 2.83} \\
Dice + CE + sL1-ACE & 5.26 ± 2.48$^{**}$ & 1.14 ± 1.59 & 3.44 ± 6.65 & 2.99 ± 8.38 \\
CE Only & 6.79 ± 3.75 & 1.25 ± 1.91 & 3.00 ± 5.68 & 2.78 ± 8.73 \\
Dice + CE + Ts & 5.10 ± 2.37$^{**}$ & \textbf{0.97 ± 0.94$^{**}$} & \textbf{2.57 ± 3.

,Dataset,Dice + CE (Baseline),Dice + CE + hL1-ACE,Dice + CE + sL1-ACE,CE Only,Dice + CE + Ts,Dice + CE + hL1-ACE + Ts,Dice + CE + sL1-ACE + Ts
0,ACDC,2.84 ± 1.68,2.76 ± 1.64,3.19 ± 2.04,2.97 ± 1.74,2.82 ± 1.66 **,2.76 ± 1.63,3.20 ± 2.03
1,AMOS,3.55 ± 3.28,3.76 ± 3.48,4.12 ± 3.39,3.80 ± 3.41,3.55 ± 3.29,3.78 ± 3.49,4.14 ± 3.38
2,BraTS,1.70 ± 3.87,1.86 ± 4.12,2.41 ± 5.77,2.13 ± 4.86,1.67 ± 3.83 **,1.82 ± 4.09,2.34 ± 5.69
3,KiTS,2.89 ± 5.90,3.91 ± 7.66,2.66 ± 5.96,3.12 ± 6.45,2.86 ± 5.88 **,3.68 ± 7.51,2.62 ± 5.96



LaTeX Table:
\begin{table}[H]
\centering
\caption{\textbf{Comparison of area under the risk-coverage curve (AURC) ($\times 10^{-2}$, lower is better) for all datasets using baseline, CE-only models, and our auxiliary calibration loss variants (hL1-ACE, sL1-ACE), with and without temperature scaling (Ts). Statistical significance relative to baseline is indicated: $^*$p $<$ 0.05, $^{**}$p $<$ 0.01. Best result per dataset is in bold.}}
\begin{tabular}{lcccc}
\hline
Method & ACDC $\downarrow$ & AMOS $\downarrow$ & BraTS $\downarrow$ & KiTS $\downarrow$ \\
\hline
Dice + CE (Baseline) & 2.84 ± 1.68 & 3.55 ± 3.28 & 1.70 ± 3.87 & 2.89 ± 5.90 \\
Dice + CE + hL1-ACE & \textbf{2.76 ± 1.64} & 3.76 ± 3.48 & 1.86 ± 4.12 & 3.91 ± 7.66 \\
Dice + CE + sL1-ACE & 3.19 ± 2.04 & 4.12 ± 3.39 & 2.41 ± 5.77 & 2.66 ± 5.96 \\
CE Only & 2.97 ± 1.74 & 3.80 ± 3.41 & 2.13 ± 4.86 & 3.12 ± 6.45 \\
Dice + CE + Ts & 2.82 ± 1.66$^{**}$ & \textbf{3.55 ± 3.29} & \textbf{1.67 ± 3.83$^{**}$} & 2.86 ± 5.88$^{**}$ \\
Dice 

\section*{AURC for Voxel-wise Selective Classification}
    
    \paragraph{Area Under the Risk--Coverage Curve (AURC).}
    To quantify how well model confidence separates correct from incorrect voxel-wise predictions, we compute the \textit{Area Under the Risk--Coverage Curve} (AURC)~\cite{geifman2017selective}.  
    AURC is a standard metric for selective classification, where a model may abstain from low-confidence predictions.  
    A lower AURC indicates better calibration and a stronger separation between confident correct predictions and confident errors.
    
    \paragraph{Per-class conditional AURC for segmentation.}
    Classical AURC computes the risk--coverage behaviour over all samples.  
    However, in semantic segmentation the model outputs multiple channels corresponding to anatomical classes, which complicates global evaluation.  
    Therefore, we compute a \textit{per-class conditional AURC}:
    
    \begin{quote}
    For each class $c$, we consider only voxels for which the model predicts class $c$ (i.e., $\arg\max$ equals $c$), sort these voxels by their predicted confidence $p_c$, and measure how well confidence discriminates correct from incorrect predictions.
    \end{quote}
    
    This evaluation yields an AURC value specific to each class and avoids coupling between classes.
    
    \paragraph{Risk--coverage curve construction.}
    For a given class $c$, AURC is computed as follows:
    
    \begin{enumerate}
        \item \textbf{Select samples.}
        \begin{itemize}
            \item In multi-class segmentation, we use all voxels for which the predicted class is $c$.
            \item If foreground-only evaluation is enabled, we further restrict to voxels where the ground-truth label is not background.
        \end{itemize}
    
        \item \textbf{Sort voxels by confidence.}  
        Voxels are ranked in descending order of the model's predicted probability $p_c$.
    
        \item \textbf{Compute risk at each coverage level.}  
        Let $e_i = 1$ if voxel $i$ is misclassified and $e_i = 0$ otherwise.  
        After sorting by confidence, the risk and coverage at step $k$ are:
        \[
            \mathrm{risk}(k) = \frac{1}{k} \sum_{i=1}^{k} e_i, \qquad
            \mathrm{coverage}(k) = \frac{k}{N},
        \]
        where $N$ is the total number of voxels predicted as class $c$.
    
        \item \textbf{Compute area under the curve.}  
        The AURC is defined as:
        \[
            \mathrm{AURC} = \int_{0}^{1} \mathrm{risk}(\mathrm{coverage}) \, d(\mathrm{coverage}),
        \]
        which we approximate using the trapezoidal rule.
    \end{enumerate}
    
    AURC is $0$ for a model that perfectly separates correct from incorrect predictions (i.e., zero risk for all coverage levels).  
    Higher AURC values correspond to poorer confidence--error separation.